# Test — 12 classifiers

**Trạng thái:** `FROZEN_OUTPUT_VERIFIED`  
**Chế độ mặc định:** CPU · read-only · không train · không resume · không chạy test inference.  

Đọc kết quả test M4 đã lưu; không chạy lại inference, không dùng kết quả này để chọn model.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Output

Chỉ đọc report test gốc, kiểm tra schema và tổ hợp 4×3.

In [3]:
TEST_OUTPUT=INPUT_CONFIG['test']  # Original M4 test ZIP or directory
if TEST_OUTPUT:
    summary=read_test_summary(TEST_OUTPUT)
    seeds=read_test_by_seed(TEST_OUTPUT)
    print('Summary:',len(summary),'rows | individual jobs:',len(seeds))
else:
    summary=[];seeds=[]
    print('SKIP: test report not attached')

Summary: 4 rows | individual jobs: 12


## Bảng số liệu

Giữ nguyên số liệu và định nghĩa chỉ số của test đã chạy.

In [4]:
if summary:
    for row in summary:
        print(row)
    out=export_json(ROOT,'test','summary.json',{'source':str(TEST_OUTPUT),'rows':summary})
    print('Saved:',out.relative_to(ROOT))
else:print('No report, no metrics generated')

{'arm': 'original_gan', 'method': 'Original GAN', 'seeds': '3', 'OA_mean': '0.8755207230106615', 'OA_sd': '0.004444846638575741', 'AA_mean': '0.8578338872200325', 'AA_sd': '0.016414870367802446', 'Macro_F1_mean': '0.8166316498653167', 'Macro_F1_sd': '0.028774505510742253', 'GM_mean': '0.8208590475054495', 'GM_sd': '0.04026838092658939', 'Kappa_mean': '0.8587124654545596', 'Kappa_sd': '0.005265954113372028'}
{'arm': 'matched_ros', 'method': 'Matched ROS', 'seeds': '3', 'OA_mean': '0.8714255454352892', 'OA_sd': '0.004220474231131371', 'AA_mean': '0.8504945335284582', 'AA_sd': '0.00681376508659191', 'Macro_F1_mean': '0.8148500245084306', 'Macro_F1_sd': '0.009673745511011864', 'GM_mean': '0.8057500335199065', 'GM_sd': '0.032694742077331286', 'Kappa_mean': '0.8541944280747537', 'Kappa_sd': '0.004496366744050772'}
{'arm': 'random_local', 'method': 'Random-local', 'seeds': '3', 'OA_mean': '0.8801807526653956', 'OA_sd': '0.011986689829380323', 'AA_mean': '0.8601562753990389', 'AA_sd': '0.01061

## Phân tích CPU

Tính lại trung bình, sample SD và hiệu số từng seed từ output test gốc. Không chấm lại classifier.

In [5]:
if TEST_OUTPUT:
    metrics = metrics_by_seed(TEST_OUTPUT)
    saved = read_test_summary(TEST_OUTPUT)
    calculated = recompute_summary(metrics)
    assert check_against_stored(calculated,saved)
    paired = test_paired_deltas(metrics)
    for row in paired:print(row)
    print('Matched metrics PASS:',len(metrics),'models')
    export_json(ROOT,'test','paired.json',{'source':str(TEST_OUTPUT),'comparisons':paired})
else:
    print('SKIP: no test output')

{'seed': 42, 'against': 'original_gan', 'macro_f1_pp': 2.5812310351530976, 'oa_pp': 1.5886464732048355}
{'seed': 42, 'against': 'matched_ros', 'macro_f1_pp': 6.061666911887609, 'oa_pp': 2.9866553696250797}
{'seed': 42, 'against': 'random_local', 'macro_f1_pp': 2.103497517017927, 'oa_pp': 0.4660029654734221}
{'seed': 43, 'against': 'original_gan', 'macro_f1_pp': -1.2427255830177009, 'oa_pp': -0.0847278119042616}
{'seed': 43, 'against': 'matched_ros', 'macro_f1_pp': -0.8917053145170106, 'oa_pp': -0.06354585892819342}
{'seed': 43, 'against': 'random_local', 'macro_f1_pp': -2.0761988511685425, 'oa_pp': -0.7837322601143781}
{'seed': 44, 'against': 'original_gan', 'macro_f1_pp': 2.6976508078454975, 'oa_pp': 0.5930946833297979}
{'seed': 44, 'against': 'matched_ros', 'macro_f1_pp': -0.599317730323845, 'oa_pp': 0.4024571065452176}
{'seed': 44, 'against': 'random_local', 'macro_f1_pp': 2.321114941950886, 'oa_pp': 1.0167337428510836}
Matched metrics PASS: 12 models


## Giới hạn

Kết quả thuộc fixed within-scene benchmark. Không coi là test không phụ thuộc không gian.

In [6]:
print('12 models: 4 arms × seeds 42, 43, 44; 4,721 fixed test centers')
print('Source: src/frozen/test.py | output: outputs/test/summary.json when inputs exist')

12 models: 4 arms × seeds 42, 43, 44; 4,721 fixed test centers
Source: src/frozen/test.py | output: outputs/test/summary.json when inputs exist


## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [7]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS
